## **ENTRENAMIENTO CON REFCOCOG**

En este fichero se evalúa el rendimiento de los modleos SAM 1 Base y SAM 1 Large tras haberlos entrenado usando el dataset RefCOCOg.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas y de medición de inferencia. Para ello se tuvo que añadir la raíz del proyecto al path de Python para que estos imports funcionen correctamente. Finalmente, se define la ruta al datset con el que entrenaremos los modelos.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from segment_anything import sam_model_registry, SamPredictor
from PIL import Image
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
import numpy as np
import os
import cv2
import pandas as pd
import time
import sys
import torch
import shutil
import random
import torch.nn as nn
import pickle


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

from utils.segmentation_quality_metrics import boundary_iou, hausdorff_95, compute_all_metrics
from utils.efficiency_metrics import measure_inference_fine_tuning_refcocog

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"


En primer lugar, se definen las rutas al dataset y los parámetros del subconjunto que se va a usar: 1000 imágenes divididas en train, val y test (70, 15, 15).

La función split_dataset carga las referencias del fichero refs(umd).p y filtra únicamente las del split de entrenamiento original de RefCOCOg. A continuación selecciona aleatoriamente un subconjunto de 1000 imágenes con una semilla fija, lo divide en tres splits y copia las imágenes a sus carpetas correspondientes, generando y guardando también la máscara de cada imagen. Los ficheros se nombran por ann_id en lugar del nombre de imagen original, ya que una misma imagen puede contener múltiples referencias a objetos distintos y cada par imagen-máscara debe poder identificarse de forma inequívoca.

In [ ]:
DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
IMAGES_DIR   = os.path.join(DATASET_ROOT, "train2014")
INSTANCES    = os.path.join(DATASET_ROOT, "instances.json")
SUBSET_SIZE  = 1000
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15

def split_dataset():
    with open(os.path.join(DATASET_ROOT, "refs(umd).p"), "rb") as f:
        refs = pickle.load(f)
    coco = COCO(INSTANCES)

    random.seed(42)
    train_refs = [r for r in refs if r["split"] == "train"]
    random.shuffle(train_refs)
    subset = train_refs[:SUBSET_SIZE]

    n = len(subset)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits = {
        "train": subset[:n_train],
        "val":   subset[n_train:n_train + n_val],
        "test":  subset[n_train + n_val:]
    }

    for split, split_refs in splits.items():
        os.makedirs(os.path.join(DATASET_ROOT, "images", split), exist_ok=True)
        os.makedirs(os.path.join(DATASET_ROOT, "masks",  split), exist_ok=True)

        for ref in split_refs:
            ann = coco.loadAnns(ref["ann_id"])[0]
            img_info = coco.loadImgs(ref["image_id"])[0]

            src = os.path.join(IMAGES_DIR, img_info["file_name"])
            if not os.path.exists(src):
                continue

            ann_id = ref["ann_id"]
            dst_img  = os.path.join(DATASET_ROOT, "images", split, f"{ann_id}.jpg")
            dst_mask = os.path.join(DATASET_ROOT, "masks",  split, f"{ann_id}.png")

            shutil.copy(src, dst_img)

            mask = coco.annToMask(ann) * 255
            Image.fromarray(mask.astype(np.uint8)).save(dst_mask)

split_dataset()

En primer lugar, se habilita TF32 para las operaciones matriciales, ya que esto nos permite acelerar el entrenamiento en GPUs NVIDIA modernas con una pérdida de precisión mínima. Además, se define el dispositivo donde se ejecutará el entrenamiento y se comprueba la disponibilidad de CUDA.

A continuación, se define el dataset cargando en el constructor las rutas de las imágenes y máscaras del split correspondiente (train, val o test). También, se filtran aquellas muestras cuya máscara no existe o está vacía (sin píxeles activos), ya que no aportarían información útil al entrenamiento.

Por último, en el método __getitem__ se aplica el preprocesamiento de cada muestra. La imagen se redimensiona a 1024x1024, se normaliza entre 0 y 1 y se convierte a tensor. Por otro lado, la máscara se redimensiona a 256x256 (resolución de salida del mask decoder de SAM) y se binariza. Finalmente, el punto central se calcula directamente a partir de los píxeles activos de la máscara en su resolución original y sus coordenadas se escalan proporcionalmente al tamaño de entrada del modelo. En caso de que la máscara esté vacía, se usa el centro de la imagen como valor por defecto.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1024):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        for img_file in os.listdir(self.images_dir):
            if not img_file.endswith(".jpg"):
                continue
            name      = os.path.splitext(img_file)[0]
            img_path  = os.path.join(self.images_dir, img_file)
            mask_path = os.path.join(self.masks_dir, name + ".png")
            if not os.path.exists(mask_path):
                continue
            gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if gt is None or (gt > 127).sum() == 0:
                continue
            self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        gt_full = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        orig_h, orig_w = gt_full.shape
        ys, xs = np.where(gt_full > 127)
        if len(xs) > 0:
            cx = float(xs.mean()) * (self.img_size / orig_w)
            cy = float(ys.mean()) * (self.img_size / orig_h)
        else:
            cx, cy = self.img_size / 2, self.img_size / 2

        mask = cv2.resize(gt_full, (256, 256))
        mask_tensor = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        point = torch.tensor([[cx, cy]]).float()
        label = torch.tensor([1])
        return image, mask_tensor, point, label


Esta función contiene el bucle de entrenamiento. En primer lugar, se carga el modelo SAM con los pesos preentrenados y se congelan tanto el image encoder como el prompt encoder, de forma que durante el entrenamiento solo se actualizan los pesos del mask decoder.

Como optimizador se usa Adam con un learning rate de 1e-4, y se añade un scheduler que reduce el learning rate a la mitad cada 20 épocas, ayudando así a afinar el modelo conforme avanza el entrenamiento. Además, se usa BCEWithLogitsLoss como función de pérdida, ya que es adecuada para segmentación binaria.

Por último, en cada época se itera sobre los batches del dataset. Para cada batch, el image encoder y el prompt encoder se ejecutan sin calcular gradientes (debido a que están congelados), mientras que el mask decoder sí los calcula. La pérdida se promedia sobre las imágenes del batch, se hace backpropagation, se actualizan los pesos y al final de cada época el scheduler ajusta el learning rate.

In [ ]:
def train_sam(dataset_path, weights_path, output_weights, vit, epochs=50, batch_size=4):
    sam = sam_model_registry[vit](checkpoint=weights_path)
    sam.to(device)

    for param in sam.image_encoder.parameters():
        param.requires_grad = False
    for param in sam.prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam.mask_decoder.parameters(), lr=1e-4)
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sam.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            images, masks  = images.to(device), masks.to(device)
            points, labels = points.to(device), labels.to(device)

            loss_batch = 0
            with torch.no_grad():
                image_embeddings = sam.image_encoder(images)

            for i in range(images.shape[0]):
                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam.prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                low_res_masks, _ = sam.mask_decoder(
                    image_embeddings=image_embeddings[i].unsqueeze(0),
                    image_pe=sam.prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save(sam.state_dict(), output_weights)
    return output_weights

**SAM 1 BASE ENTRENAMIENTO**

En primer lugar, se definen las rutas a los distintos recursos que se van a usar: el dataset, los pesos preentrenados y las rutas donde se guardarán los resultados y los pesos del modelo entrenado. En este caso, se procede a evaluar el rendimiento del modelo SAM 1 Base.

Respecto a la función de evaluación, esta evalúa el modelo fine-tuneado sobre el conjunto de test. Para cada referencia del split de test se carga la anotación individual desde la API de COCO, se genera su máscara con coco.annToMask y se descarta si está vacía. El prompt utilizado para la inferencia es la caja delimitadora de la anotación, igual que en la evaluación zero-sho, permitiendo así la comparación entre ambas. La máscara predicha se redimensiona al tamaño original de la imagen para que sea comparable con el ground truth, y se calculan todas las métricas. Al final se devuelve un diccionario con la media de cada métrica sobre todo el conjunto de test.

Finalmente, se lanza el entrenamiento midiendo el tiempo que tarda, y después se evalúa el modelo resultante, midiendo también su tiempo. Ambos tiempos se añaden al diccionario de resultados, que finalmente se guarda en un CSV. Si el fichero ya existe se añade una nueva línea sin sobreescribir los resultados anteriores, lo que permite ir acumulando los resultados de distintos experimentos en el mismo fichero.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"
print(f"Usando GPU: {torch.cuda.get_device_name(0)}")

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam_b_refcocog.pt"


def evaluate_model(dataset, weights_path):
    sam = sam_model_registry["vit_b"](checkpoint=weights_path)
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    with open(os.path.join(dataset, "refs(umd).p"), "rb") as f:
        refs = pickle.load(f)
    coco = COCO(os.path.join(dataset, "instances.json"))
    refs_test = [r for r in refs if r["split"] == "test"]

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "train2014")
    
    for ref in refs_test:
        ann = coco.loadAnns(ref["ann_id"])[0]
        img_info = coco.loadImgs(ref["image_id"])[0]
        img_path = os.path.join(images_dir, img_info["file_name"])

        gt_mask = coco.annToMask(ann).astype(bool)
        if gt_mask.sum() == 0:
            continue

        x, y, w, h = ann["bbox"]
        bbox = np.array([x, y, x + w, y + h])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks, scores, latency, vram = measure_inference_fine_tuning_refcocog(predictor, image, bbox)

        if masks is None or len(masks) == 0:
            continue

        H, W = gt_mask.shape
        best_idx = np.argmax(scores)
        pred_mask = cv2.resize(masks[best_idx].astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_b_refcocog"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results
         

start_train = time.time()
trained_weights = train_sam(dataset, weights, output_weights, vit="vit_b")
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(dataset, trained_weights)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Usando GPU: NVIDIA GeForce RTX 3090


c:\users\danieltalavera\desktop\trabajo_fin_de_grado\segment-anything\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.loa

Epoch 1/50 - Loss: 0.4423
Epoch 2/50 - Loss: 0.3269
Epoch 3/50 - Loss: 0.2988
Epoch 4/50 - Loss: 0.2875
Epoch 5/50 - Loss: 0.2866
Epoch 6/50 - Loss: 0.2648
Epoch 7/50 - Loss: 0.2570
Epoch 8/50 - Loss: 0.2501
Epoch 9/50 - Loss: 0.2388
Epoch 10/50 - Loss: 0.2436
Epoch 11/50 - Loss: 0.2282
Epoch 12/50 - Loss: 0.2133
Epoch 13/50 - Loss: 0.2161
Epoch 14/50 - Loss: 0.2098
Epoch 15/50 - Loss: 0.2044
Epoch 16/50 - Loss: 0.2103
Epoch 17/50 - Loss: 0.1975
Epoch 18/50 - Loss: 0.1850
Epoch 19/50 - Loss: 0.1950
Epoch 20/50 - Loss: 0.1851
Epoch 21/50 - Loss: 0.1694
Epoch 22/50 - Loss: 0.1576
Epoch 23/50 - Loss: 0.1513
Epoch 24/50 - Loss: 0.1474
Epoch 25/50 - Loss: 0.1434
Epoch 26/50 - Loss: 0.1423
Epoch 27/50 - Loss: 0.1405
Epoch 28/50 - Loss: 0.1411
Epoch 29/50 - Loss: 0.1355
Epoch 30/50 - Loss: 0.1341
Epoch 31/50 - Loss: 0.1342
Epoch 32/50 - Loss: 0.1303
Epoch 33/50 - Loss: 0.1289
Epoch 34/50 - Loss: 0.1337
Epoch 35/50 - Loss: 0.1320
Epoch 36/50 - Loss: 0.1251
Epoch 37/50 - Loss: 0.1194
Epoch 38/5

**SAM 1 LARGE ENTRENAMIENTO**

A continuación, se replica el procedimiento aplicado en el bloque de código anterior con la única diferencia de que el modelo a evaluar en este caso será SAM 1 Large. De esta manera podremos comparar las diferencias de rendimiento entre las distintas versiones de cada modelo.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"
print(f"Usando GPU: {torch.cuda.get_device_name(0)}")

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam_l_refcocog.pt"


def evaluate_model(dataset, weights_path):
    sam = sam_model_registry["vit_l"](checkpoint=weights_path)
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    with open(os.path.join(dataset, "refs(umd).p"), "rb") as f:
        refs = pickle.load(f)
    coco = COCO(os.path.join(dataset, "instances.json"))
    refs_test = [r for r in refs if r["split"] == "test"]

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "train2014")
    
    for ref in refs_test:
        ann = coco.loadAnns(ref["ann_id"])[0]
        img_info = coco.loadImgs(ref["image_id"])[0]
        img_path = os.path.join(images_dir, img_info["file_name"])

        gt_mask = coco.annToMask(ann).astype(bool)
        if gt_mask.sum() == 0:
            continue

        x, y, w, h = ann["bbox"]
        bbox = np.array([x, y, x + w, y + h])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        masks, scores, latency, vram = measure_inference_fine_tuning_refcocog(predictor, image, bbox)

        if masks is None or len(masks) == 0:
            continue

        H, W = gt_mask.shape
        best_idx = np.argmax(scores)
        pred_mask = cv2.resize(masks[best_idx].astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_l_refcocog"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results
         

start_train = time.time()
trained_weights = train_sam(dataset, weights, output_weights, vit="vit_l")
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(dataset, trained_weights)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Usando GPU: NVIDIA GeForce RTX 3090
Epoch 1/50 - Loss: 0.4406
Epoch 2/50 - Loss: 0.3085
Epoch 3/50 - Loss: 0.2761
Epoch 4/50 - Loss: 0.2553
Epoch 5/50 - Loss: 0.2420
Epoch 6/50 - Loss: 0.2260
Epoch 7/50 - Loss: 0.2187
Epoch 8/50 - Loss: 0.2047
Epoch 9/50 - Loss: 0.1960
Epoch 10/50 - Loss: 0.1856
Epoch 11/50 - Loss: 0.1775
Epoch 12/50 - Loss: 0.1757
Epoch 13/50 - Loss: 0.1681
Epoch 14/50 - Loss: 0.1595
Epoch 15/50 - Loss: 0.1595
Epoch 16/50 - Loss: 0.1552
Epoch 17/50 - Loss: 0.1481
Epoch 18/50 - Loss: 0.1665
Epoch 19/50 - Loss: 0.1492
Epoch 20/50 - Loss: 0.1440
Epoch 21/50 - Loss: 0.1297
Epoch 22/50 - Loss: 0.1211
Epoch 23/50 - Loss: 0.1169
Epoch 24/50 - Loss: 0.1136
Epoch 25/50 - Loss: 0.1111
Epoch 26/50 - Loss: 0.1100
Epoch 27/50 - Loss: 0.1092
Epoch 28/50 - Loss: 0.1071
Epoch 29/50 - Loss: 0.1050
Epoch 30/50 - Loss: 0.1035
Epoch 31/50 - Loss: 0.1019
Epoch 32/50 - Loss: 0.1013
Epoch 33/50 - Loss: 0.1035
Epoch 34/50 - Loss: 0.1000
Epoch 35/50 - Loss: 0.0977
Epoch 36/50 - Loss: 0.0949
E